# 06. DiD를 위한 EDA — 실무 7단계

| 단계 | 하는 일 | 실무에서 자주 하는 말 |
|---|---|---|
| 0. 질문 고정 | 의사결정 질문, 분석 단위, 결과 지표, 비교 대상, 판정 규칙을 먼저 적음 | "무엇을 결정하려고 보는가?" |
| 1. 데이터 이해 | 출처, 수집 방식(표본조사, 가중치, 오차), 한 행의 단위, 기간 | "이 숫자는 어떻게 만들어졌나?" |
| 2. 품질 점검 | 행 수, 키 중복, 결측, 값 범위, 공표값과 맞춰보기 | "믿기 전에 맞춰본다" |
| 3. 한 변수씩 | 분포, 치우침, 이상치 → 변환 결정, 처치 전 균형 | "평균 말고 분포를 봐라" |
| 4. 관계 | 결과와 처치의 관계, 시간 추세, 집단별 비교 | "합계만 보지 말고 쪼개 봐라" |
| 5. 다른 설명 찾기 | 같은 시기의 다른 충격, 구성 변화, 교란요인 | "AI 말고 이걸 설명할 게 뭐가 있나?" |
| 6. 결정 기록 | 결과변수, 표본, 기간, 모형을 정하고 이유를 남김 | "EDA의 산출물은 그래프가 아니라 결정" |

- 입력: `data/processed/panel_analysis.csv`(01 노트북), `data/raw/kosis/DT_1DA7002S_*.csv`(월별 전국). **공개 자료만 쓴다**(팀원 누구나 실행 가능).
- 출력: `data/processed/did_treatment.csv`, `reports/did_eda_results.csv`, `reports/figures/fig15~19`

**통계적 유의성 읽는 법**: 모든 검정은 양측, 유의수준 5%다. 출력의 `판정` 열은 p값으로 자동으로 붙는다
(p<0.01 "매우 유의", p<0.05 "유의", p<0.10 "경계", 그 밖 "유의하지 않음").
"유의하지 않음"은 "효과가 없다"가 아니라 "이 자료로는 0과 구분되지 않는다"는 뜻이다. 그래서 신뢰구간의 폭도 함께 본다.

In [ ]:
import sys
import warnings

sys.path.append("..")  # 노트북 폴더 기준으로 src를 찾는다 (README 6절)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from matplotlib import font_manager
from scipy import stats

from src import config

warnings.filterwarnings("ignore", category=FutureWarning)
INK2, MUTED, GRID, AXIS, SURFACE = "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
BLUE, ORANGE, TERC = "#2a78d6", "#eb6834", {"하": "#86b6ef", "중": "#2a78d6", "상": "#0d366b"}
_fonts = {f.name for f in font_manager.fontManager.ttflist}
FONT = next(f for f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "DejaVu Sans"] if f in _fonts)
sns.set_theme(style="whitegrid", font=FONT, rc={
    "axes.unicode_minus": False, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.edgecolor": AXIS, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.labelsize": 9,
    "axes.titleweight": "bold", "axes.titlesize": 10, "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})
LEVELS = ["하", "중", "상"]
REF, START = "202202", "201901"   # 기준 시점(ChatGPT 직전 조사), 주 분석 시작
results = []                       # 6단계 결정표에 쓸 검정 결과


def sig(p):
    """p값 → 판정 문구 (양측, 유의수준 5%)."""
    return "매우 유의 (p<0.01)" if p < 0.01 else "유의 (p<0.05)" if p < 0.05 else "경계 (p<0.10)" if p < 0.10 else "유의하지 않음"


def fit(d, terms, y="lr_y", trend=False, fe="code"):
    """DiD 회귀: 가중치 = 2022년 청년 취업자, 표준오차는 직업 단위 군집(50개).

    fe="code"면 직업·시점 고정효과, fe="major"면 시점 고정효과를 대분류마다 따로 둔다(같은 대분류 안에서만 비교).
    trend=True면 직업별 직선 추세를 더한다(하나는 빼야 시점 더미와 겹치지 않음).
    """
    d = d.dropna(subset=terms + [y])
    parts = [d[terms], pd.get_dummies(d.code, prefix="c", drop_first=True)]
    if fe == "major":
        mp = pd.get_dummies(d.major.astype(str) + "_" + d.period, prefix="mp")
        parts.append(mp.loc[:, ~mp.columns.str.endswith(d.period.min())])   # 대분류마다 첫 시점을 빼야 직업 더미와 겹치지 않음
    else:
        parts.append(pd.get_dummies(d.period, prefix="p", drop_first=True))
    if trend:
        parts.append(pd.get_dummies(d.code, prefix="tr", drop_first=True).mul(d.t, axis=0))
    X = sm.add_constant(pd.concat(parts, axis=1).astype(float))
    assert np.linalg.matrix_rank(X.values) == X.shape[1], "설명변수끼리 겹침"
    return sm.WLS(d[y], X, weights=d.w).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.code)[0]})


def report(name, m, term, stage, note=""):
    b, (lo, hi), p = m.params[term], m.conf_int().loc[term], m.pvalues[term]
    r = {"단계": stage, "분석": name, "효과(%)": 100 * b, "하한": 100 * lo, "상한": 100 * hi, "p": p, "판정": sig(p),
         "관측": int(m.nobs), "비고": note}
    results.append(r)
    print(f"{name}: {100 * b:+.2f}% [{100 * lo:+.2f}, {100 * hi:+.2f}], p={p:.3f} → {sig(p)}")
    return r

## 0. 질문 고정

결과를 보기 전에 정해야 하는 것들이다. 다만 이 팀은 이미 03~05 노트북에서 결과 일부를 봤다
(노출 직업의 청년 감소가 2013~2019년에도 있었음). 그래서 아래 규칙은 **그 사실을 알고 정했다**는 점을 밝혀 둔다.

| 항목 | 정한 것 |
|---|---|
| 의사결정 질문 | 생성형 AI 출시(2022.11) 이후, AI 노출이 큰 직업에서 청년 취업자가 노출이 작은 직업보다 **더** 줄었는가? |
| 분석 단위 | 직업 중분류 × 반기 (주 분석 50개 직업) |
| 결과 지표 | 주: `lr_y` = ln(청년 ÷ 30~59세). 함께: `ly` = ln(청년 취업자). 직업별 고용률은 정의할 수 없음(직업별 인구가 없음) |
| 처치 | ILO **2023** 점수. 회귀는 연속값(1표준편차당), 그림은 상위 1/3 vs 하위 1/3 |
| 비교 대상 | 노출도 하위 1/3 직업 (연속값 회귀에서는 노출도가 낮은 직업 전체) |
| 시점 | 처치 전 2019 상 ~ 2022 하(8개 반기), 처치 후 2023 상 ~ 2026 상(7개 반기). 2013~2018은 추세 확인용 |
| 추론 | 가중치 = 2022년 청년 취업자, 표준오차는 직업 단위 군집, 양측 5% |

**판정 규칙**

- **R1 평행추세**: 처치 전 반기별 계수의 합동 검정 p < 0.05면 평행추세가 깨진 것으로 보고, 단순 DiD 대신 직업별 추세를 넣은 결과를 주 결과로 쓴다.
- **R2 청년만의 변화인가**: 30~59세도 고노출 직업에서 유의하게 줄었으면, 청년 문제가 아니라 직업 전체의 변화로 본다.
- **R3 견고성**: 효과를 말하려면 주 결과가 유의하고, 2025 점수와 직업 하나씩 뺀 결과에서도 부호가 같아야 한다.
- **R4 위약 검정**: AI 이전 가짜 시점(2019 상)에 같은 분석을 했을 때 유의하면, 같은 크기의 효과가 AI 없이도 나온다는 뜻이다.

## 1. 데이터 이해

| 자료 | 출처·수집 방식 | 한 행 | 기간 | 주의 |
|---|---|---|---|---|
| 직업별 취업자 `DT_1ES3B28J` | 통계청 **지역별고용조사**(표본조사, 반기마다 약 23만 가구) 공표값 | 직업 중분류 × 연령 × 반기 | 2013 상 ~ 2026 상 | 상반기는 4월, 하반기는 10월 조사. 표본 추정치라 작은 칸은 오차가 큼(`*` 표시) |
| 전국 월별 `DT_1DA7002S` | 통계청 **경제활동인구조사**(매달 약 3.5만 가구) | 연령 × 월 | 2013.01 ~ 2026.10 | 직업별 표와 다른 조사지만, 아래 2-2에서 4월·10월 값이 0.2% 안에서 맞는다 |
| AI 노출도 | ILO WP140의 2023·2025 점수(ISCO 4자리) → 통계청 연계표로 한국 세분류 → 중분류 평균 | 직업 | 고정값 | 업무의 **기술적** 노출 가능성. 실제 AI 사용률이나 감소 예측이 아님 |

In [ ]:
p = pd.read_csv(config.DATA_PROCESSED / "panel_analysis.csv", dtype={"code": str, "period": str})
PERIODS = sorted(p.period.unique())
p["t"] = p.time - p.loc[p.period == REF, "time"].iloc[0]   # 2022 하반기 = 0 (연 단위)
p["ly"], p["la"] = np.log(p.y15_29.where(p.y15_29 > 0)), np.log(p.a30_59)
print(f"전체 {p.shape[0]:,}행 × {p.shape[1]}열, 직업 {p.code.nunique()}개, 시점 {len(PERIODS)}개 ({PERIODS[0]} ~ {PERIODS[-1]})")
print("주 분석 직업:", p[p.in_main].code.nunique(), "| 단위: 천 명 | 대분류별 직업 수:", p[p.in_main].drop_duplicates("code").major.value_counts().sort_index().to_dict())

raw = pd.read_csv(next((config.DATA_RAW / "kosis").glob("DT_1DA7002S_*.csv")), encoding="cp949", header=None, dtype=str)
per, item = raw.iloc[0, 1:].str[1:7], raw.iloc[1, 1:].str[:3]


def series(age, it):
    r = raw[raw[0].str.startswith(age)].iloc[0, 1:]
    s = pd.to_numeric(r[(item == it).values], errors="coerce")
    s.index = pd.to_datetime(per[(item == it).values].values, format="%Y%m")
    return s

## 2. 품질 점검

### 2-1. 구조 검사 — 하나라도 어긋나면 멈춘다

In [ ]:
main = p[p.in_main].copy()
checks = {
    "(직업, 시점) 중복 없음": not p.duplicated(["code", "period"]).any(),
    "주 분석 50개 직업 × 27개 시점 = 1,350행": len(main) == 1350,
    "주 분석에 결과변수 빈칸 없음": bool(np.isfinite(main[["lr_y", "ly", "la"]]).all().all()),
    "취업자 음수 없음, 청년 ≤ 전체": bool((p[["y15_29", "a30_59", "total"]] >= 0).all().all() and (p.y15_29 <= p.total).all()),
    "노출도 0~1": bool(p[["exp23", "exp25"]].stack().between(0, 1).all()),
}
for k, v in checks.items():
    print(("통과  " if v else "실패  ") + k)
assert all(checks.values())

### 2-2. 공표값과 맞춰보기

직업별 표(지역별고용조사)의 청년 취업자 합계를, 같은 달의 전국 월별 표(경제활동인구조사)와 비교한다.
중요한 것은 **차이가 시간에 따라 일정한가**다. 차이가 한쪽으로 커지면 직업별 표의 추세를 의심해야 한다.

통계 검정: 차이(%)를 시간에 회귀해 기울기가 0인지 본다.

In [ ]:
nat = series("75 ", "T30")
occ_sum = p.groupby("period").y15_29.sum()
mon = {1: 4, 2: 10}
cmp = pd.DataFrame({"직업별 합계": occ_sum,
                    "전국 월별": [nat[pd.Timestamp(int(k[:4]), mon[int(k[5])], 1)] for k in occ_sum.index]})
cmp["차이(%)"] = 100 * (cmp["직업별 합계"] / cmp["전국 월별"] - 1)
print(cmp.round(1).T.to_string())
tt = np.arange(len(cmp)) / 2
lr = stats.linregress(tt, cmp["차이(%)"])
print(f"\n차이의 연간 변화 {lr.slope:+.2f}%p/년, p={lr.pvalue:.3f} → {sig(lr.pvalue)}")
print(f"두 조사의 반기 변화율 상관: {np.corrcoef(np.diff(np.log(cmp['직업별 합계'])), np.diff(np.log(cmp['전국 월별'])))[0, 1]:.2f}")

### 2-3. 오차가 큰 칸이 한 집단에 몰리는가

KOSIS는 청년 추정치의 상대표준오차가 25% 이상이면 `*`를 붙인다(`y_flag`). 처치 후에 비교 집단에서만 이런 칸이 늘면 그 집단의 추세가 흐려진다.
통계 검정: 집단마다 처치 전·후 오차 표시 비율이 같은지 피셔 정확 검정(칸 수가 적어서 카이제곱 대신).
처치 집단은 아래 3-1에서 정하므로, 여기서는 2023 점수 3분위를 먼저 만든다.

In [ ]:
occ = main.drop_duplicates("code")[["code", "name", "major", "exp23", "exp25", "w"]].reset_index(drop=True)
for s in ("23", "25"):
    e = occ[f"exp{s}"]
    occ[f"z{s}"] = (e - e.mean()) / e.std()
    occ[f"t{s}"] = pd.qcut(e.rank(method="first"), 3, labels=LEVELS).astype(str)   # 동점은 순서대로 17·16·17개
    occ[f"treat{s}"] = occ[f"t{s}"].map({"상": 1.0, "하": 0.0})
main = main.merge(occ[["code", "z23", "z25", "t23", "t25", "treat23", "treat25"]], on="code")
main["post"] = main.year >= 2023

fl = []
for t in LEVELS:
    g = main[(main.t23 == t) & (main.year >= 2019)]
    tab = pd.crosstab(g.post, g.y_flag).reindex(index=[False, True], columns=[False, True], fill_value=0)
    pv = stats.fisher_exact(tab.values)[1]
    fl.append({"집단": t, "처치 전 %": 100 * g[~g.post].y_flag.mean(), "처치 후 %": 100 * g[g.post].y_flag.mean(), "p": pv, "판정": sig(pv)})
    if t == "하":
        results.append({"단계": "2", "분석": "비교 집단(하)의 오차 표시 증가", "효과(%)": fl[-1]["처치 후 %"] - fl[-1]["처치 전 %"], "p": pv, "판정": sig(pv),
                        "비고": "효과 = 오차 표시 비율 변화(%p)"})
print(pd.DataFrame(fl).round(3).to_string(index=False))
w_share = main.assign(wf=main.w * main.y_flag).groupby(["t23", "post"]).apply(lambda g: 100 * g.wf.sum() / g.w.sum(), include_groups=False)
print("\n청년 취업자 가중 기준 오차 표시 비율(%) — 회귀에서 실제로 차지하는 비중:\n" + w_share.unstack().reindex(LEVELS).round(2).to_string())

### 2-4. 단절과 계절

분류 개정이 있으면 한 시점에 여러 직업의 전체 취업자가 한꺼번에 튄다. 상반기(4월)와 하반기(10월)의 차이도 본다.
통계 검정: 같은 해 하반기 − 상반기 차이의 평균이 0인지 t검정.

In [ ]:
main = main.sort_values(["code", "period"])
main["d_ltot"] = main.groupby("code").total.transform(lambda s: np.log(s).diff())
jumps = main.groupby("period").d_ltot.apply(lambda s: int((s.abs() > 0.25).sum()))
print("전체 취업자가 한 반기에 25% 넘게 변한 직업 수가 있는 시점:", jumps[jumps > 0].to_dict() or "없음")
for c in ("lr_y", "ly"):
    s = main.pivot_table(index=["code", "year"], columns="half", values=c).dropna()
    tt_ = stats.ttest_1samp(s[2] - s[1], 0)
    print(f"{c}: 하반기 − 상반기 평균 {(s[2] - s[1]).mean():+.3f}, p={tt_.pvalue:.3f} → {sig(tt_.pvalue)}")

**2단계 해석** (2026-11-24 실행 기준)

- 직업별 합계는 27개 시점 모두 전국 월별 표와 **0.2% 안에서** 맞고, 차이가 시간에 따라 커지지도 않는다(p=0.20, 유의하지 않음). 직업별 표의 추세는 믿을 만하다.
- 비교 집단(하)에서 오차가 큰 칸이 처치 후 3.7% → 10.9%로 **유의하게** 늘었다(p=0.028). 다만 청년이 몇천 명인 작은 직업 3개 때문이라, 가중하면 비중이 1.8%에 그친다. 5단계에서 이 직업들을 빼고 다시 확인한다.
- 분류 개정 흔적은 없고, 상·하반기 차이는 **유의하지 않다**(p=0.44~0.58). 시점 고정효과로 충분하다.

## 3. 한 변수씩

### 3-1. 처치(노출도) 분포와 2023·2025 점수 비교

통계 검정: 두 점수의 순위 상관(스피어만)이 0인지.

In [ ]:
print(occ[["exp23", "exp25"]].describe().round(3).T.to_string())
rho, prho = stats.spearmanr(occ.exp23, occ.exp25)
print(f"\n순위 상관 {rho:.3f}, p={prho:.1e} → {sig(prho)}")
cross = pd.crosstab(occ.t23, occ.t25).reindex(index=LEVELS, columns=LEVELS)
print(cross.to_string())
moved = occ[occ.t23 != occ.t25]
print(f"3분위가 바뀌는 직업 {len(moved)}개:", ", ".join(moved.name))
cols = ["code", "name", "exp23", "w"]
print("\n[상위 6]\n" + occ.nlargest(6, "exp23")[cols].round(3).to_string(index=False))
print("[하위 6]\n" + occ.nsmallest(6, "exp23")[cols].round(3).to_string(index=False))
occ[["code", "name", "major", "exp23", "exp25", "z23", "z25", "t23", "t25", "treat23", "treat25", "w"]].to_csv(
    config.DATA_PROCESSED / "did_treatment.csv", index=False, encoding="utf-8-sig")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1, 1.1]})
bins = np.linspace(0.05, 0.75, 15)
a1.hist(occ.exp23, bins=bins, color=TERC["중"], alpha=0.85, label="2023 점수")
a1.hist(occ.exp25, bins=bins, histtype="step", color=ORANGE, lw=1.8, label="2025 점수")
for q in occ.exp23.quantile([1 / 3, 2 / 3]):
    a1.axvline(q, color=MUTED, lw=0.8, ls="--")
a1.set(title="노출도 분포 (50개 직업)", xlabel="ILO 생성형 AI 노출도", ylabel="직업 수")
a1.legend(frameon=False, fontsize=8)
for t in LEVELS:
    g = occ[occ.t23 == t]
    a2.scatter(g.exp23, g.exp25, s=g.w / 4 + 8, color=TERC[t], alpha=0.85, label=f"2023 기준 {t}", edgecolor="white", lw=0.5)
lim = [0.05, 0.75]
a2.plot(lim, lim, color=AXIS, lw=0.8)
for _, row in moved.iterrows():
    low = row.exp23 < 0.25   # 아래쪽 점 무리와 겹치지 않게 비어 있는 곳으로 빼고 선을 잇는다
    off = ((0, 40) if row.exp25 > row.exp23 else (30, -30)) if low else (6, -3)
    a2.annotate(row["name"], (row.exp23, row.exp25), xytext=off, textcoords="offset points", ha="center" if off[0] == 0 else "left",
                fontsize=7, color=INK2, arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6) if low else None)
a2.set(xlim=lim, ylim=lim, title=f"2023 vs 2025 점수 (순위 상관 {rho:.2f})", xlabel="2023 점수", ylabel="2025 점수")
for h in a2.legend(frameon=False, fontsize=8, loc="upper left").legend_handles:
    h.set_sizes([30])
a2.text(0.98, 0.02, "점 크기 = 2022년 청년 취업자\n이름 = 3분위 집단이 바뀌는 직업", transform=a2.transAxes, ha="right", fontsize=7, color=MUTED)
fig.tight_layout()
fig.savefig(config.FIGURES / "fig15_처치정의_노출도.png")
plt.show()

### 3-2. 결과변수 분포 — 왜 로그를 쓰는가

직업마다 청년 수가 수천 명에서 수십만 명까지 차이 난다. 수 그대로 비교하면 큰 직업이 같은 비율로 줄어도 "더 많이 줄어든" 것처럼 보인다.
통계 검정: 치우침(왜도)이 0인지(정규분포면 0). 로그를 씌운 뒤 치우침이 줄어드는지 본다.

In [ ]:
y22 = main[main.year == 2022].groupby("code").y15_29.mean()
for name, v in {"청년 취업자(천 명)": y22, "ln(청년 취업자)": np.log(y22)}.items():
    sk = stats.skewtest(v)
    print(f"{name}: 최소 {v.min():.1f}, 중앙값 {v.median():.1f}, 최대 {v.max():.1f}, 왜도 {stats.skew(v):+.2f} (p={sk.pvalue:.3f} → {sig(sk.pvalue)})")
main["d_lr_y"] = main.groupby("code").lr_y.diff()
print(f"\n반기 변화(lr_y) 분포: 표준편차 {main.d_lr_y.std():.3f}, 첨도 {stats.kurtosis(main.d_lr_y.dropna()):.1f} (0보다 크면 꼬리가 두꺼움 = 가끔 크게 튐)")

### 3-3. 처치 전 균형 — 상·하 집단은 원래 비슷했나

DiD는 두 집단의 **수준**이 같을 필요는 없지만(직업 고정효과가 지움), **원래 추세**는 같아야 한다.
통계 검정: 상(17개) vs 하(17개) 직업의 평균 차이를 웰치 t검정. 표준화 차이(SMD)가 0.25를 넘으면 실무에서는 "불균형"으로 본다.

In [ ]:
def slope(g, lo, hi):
    g = g[(g.year >= lo) & (g.year <= hi)]
    return np.polyfit(g.time, g.lr_y, 1)[0]


bal = occ.set_index("code")[["t23"]].copy()
bal["2022 ln(청년)"] = np.log(y22)
bal["2022 청년 비중(%)"] = main[main.year == 2022].groupby("code").y_share.mean()
bal["2022 lr_y"] = main[main.year == 2022].groupby("code").lr_y.mean()
bal["lr_y 추세 2013~19 (%/년)"] = 100 * main.groupby("code").apply(slope, 2013, 2019, include_groups=False)
bal["lr_y 추세 2019~22 (%/년)"] = 100 * main.groupby("code").apply(slope, 2019, 2022, include_groups=False)
rows = []
for c in bal.columns[1:]:
    h, l = bal.loc[bal.t23 == "상", c], bal.loc[bal.t23 == "하", c]
    tt_ = stats.ttest_ind(h, l, equal_var=False)
    smd = (h.mean() - l.mean()) / np.sqrt((h.var() + l.var()) / 2)
    rows.append({"변수": c, "상 평균": h.mean(), "하 평균": l.mean(), "SMD": smd, "p": tt_.pvalue, "판정": sig(tt_.pvalue)})
    if "추세" in c:
        results.append({"단계": "3", "분석": f"처치 전 균형: {c} 상−하", "효과(%)": h.mean() - l.mean(), "p": tt_.pvalue, "판정": sig(tt_.pvalue),
                        "비고": "효과 = 연간 추세 차이(%p/년), 직업 단위 단순 비교"})
print(pd.DataFrame(rows).round(3).to_string(index=False))

**3단계 해석**

- 2023·2025 점수는 순위 상관 0.97로 **매우 유의하게** 같은 직업을 고른다. 처치 정의는 점수 버전에 거의 좌우되지 않는다.
- 청년 취업자 수는 오른쪽으로 크게 치우쳐 있고(왜도 +1.94, 매우 유의), 로그를 씌우면 치우침이 **유의하지 않은** 수준(−0.47, p=0.14)으로 줄어든다. 로그를 쓰는 근거다.
- 상 집단은 하 집단보다 원래 크고 청년 비중도 높다(**매우 유의**). 수준 차이는 직업 고정효과가 지우므로 문제가 아니다.
- **문제는 추세다.** 2013~2019년에 이미 상 집단의 `lr_y`가 하 집단보다 연 4.7%p 더 빨리 떨어지고 있었다(**매우 유의**, p=0.008). 2019~2022년은 차이 2.8%p로 **유의하지 않다**(p=0.21). 평행추세가 깨질 가능성이 높다는 신호다.

## 4. 관계

### 4-1. 원자료 추세 — 집단 합계를 2022 하반기 = 100으로

회귀 전에 눈으로 먼저 본다. 청년만 보지 않고 같은 직업의 30~59세도 함께 그려야, 직업 전체가 줄어든 것인지 청년만 줄어든 것인지 구분된다.

In [ ]:
grp = main[main.t23 != "중"].groupby(["t23", "period", "time"])[["y15_29", "a30_59"]].sum().reset_index()
for c in ("y15_29", "a30_59"):
    grp[c + "_idx"] = 100 * grp[c] / grp.groupby("t23")[c].transform(lambda s: s[grp.loc[s.index, "period"] == REF].iloc[0])
print(grp.pivot(index="period", columns="t23", values=["y15_29_idx", "a30_59_idx"]).round(1).iloc[[0, 6, 12, 16, 19, 22, 26]].to_string())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, c, title in zip(axes, ("y15_29_idx", "a30_59_idx"), ("청년(15~29세) 취업자", "30~59세 취업자 (같은 직업)")):
    for t in ("상", "하"):
        g = grp[grp.t23 == t]
        ax.plot(g.time, g[c], color=TERC[t], lw=2, marker="o", ms=3, label=f"노출도 {t} ({(occ.t23 == t).sum()}개 직업)")
    ax.axvspan(2022.9, 2026.6, color=GRID, alpha=0.5, lw=0)
    ax.axhline(100, color=AXIS, lw=0.8)
    ax.text(2023.0, 0.97, "생성형 AI 이후", transform=ax.get_xaxis_transform(), fontsize=7, color=MUTED, va="top")
    ax.set(title=title, xlim=(2012.8, 2026.6))
axes[0].set_ylabel("2022 하반기 = 100")
axes[0].legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig(config.FIGURES / "fig16_노출도별_추세.png")
plt.show()

### 4-2. 2×2 표 — 가장 단순한 DiD

(상 집단의 처치 후 − 처치 전) − (하 집단의 처치 후 − 처치 전). 집단 평균은 2022년 청년 취업자로 가중한다. 이 표는 검정이 없는 기술 통계다.

In [ ]:
d19 = main[main.year >= 2019]
tbl = d19[d19.t23 != "중"].groupby(["t23", "post"]).apply(lambda g: np.average(g.lr_y, weights=g.w), include_groups=False).unstack()
tbl.columns = ["처치 전(2019~22)", "처치 후(2023~26)"]
tbl["변화"] = tbl.iloc[:, 1] - tbl.iloc[:, 0]
print(tbl.round(3).to_string())
print(f"\n단순 DiD (상 − 하): {tbl.loc['상', '변화'] - tbl.loc['하', '변화']:+.3f} ≈ {100 * (tbl.loc['상', '변화'] - tbl.loc['하', '변화']):+.1f}%")

### 4-3. 회귀 DiD — 유의한가

직업·시점 고정효과를 넣은 회귀로 같은 질문을 검정한다. 계수 × 100 ≈ % 차이.

- 연속값: 노출도가 1표준편차 높은 직업에서 처치 후 몇 % 더 변했나 (50개 직업 모두 사용)
- 이분: 상 집단이 하 집단보다 몇 % 더 변했나 (중 집단 제외, 34개 직업)

In [ ]:
d19 = d19.assign(x23=d19.z23 * d19.post, x25=d19.z25 * d19.post, xb=d19.treat23 * d19.post)
m_main = fit(d19, ["x23"])
report("주 결과: lr_y, 노출도 1SD × 처치 후", m_main, "x23", "4")
report("lr_y, 상 vs 하 × 처치 후", fit(d19.dropna(subset=["treat23"]), ["xb"]), "xb", "4")
report("ly(청년 취업자 수), 노출도 1SD × 처치 후", fit(d19, ["x23"], y="ly"), "x23", "4")

### 4-4. 이벤트 스터디 — 평행추세 검정 (R1)

반기마다 따로 "노출도 1SD당 차이"를 추정한다(기준 = 2022 하반기). 처치 전 계수가 0 근처에서 평평해야 평행추세가 성립한다.
통계 검정: 처치 전 계수가 **모두 0**인지 합동 F검정. 2019~2022(주 분석 기간)와 2013~2022(전체) 두 가지로 본다.

In [ ]:
ev_cols = [f"ev_{k}" for k in PERIODS if k != REF]
for k in PERIODS:
    if k != REF:
        main[f"ev_{k}"] = main.z23 * (main.period == k)
m_ev = fit(main, ev_cols)
pre_all = [c for c in ev_cols if c[3:] < "202301"]
pre_19 = [c for c in pre_all if c[3:] >= START]
pre_tests = {}
for name, cs in {"2019~2022": pre_19, "2013~2022": pre_all}.items():
    ft = m_ev.f_test(", ".join(f"{c} = 0" for c in cs))
    pv = float(ft.pvalue)
    pre_tests[name] = pv
    results.append({"단계": "4", "분석": f"R1 평행추세 합동검정 ({name}, {len(cs)}개 계수)", "p": pv, "판정": sig(pv),
                    "비고": "유의하면 평행추세가 깨짐"})
    print(f"처치 전 {name} 계수 {len(cs)}개가 모두 0인가: F={float(ft.fvalue):.2f}, p={pv:.3f} → {sig(pv)}"
          + ("  ← 평행추세 기각" if pv < 0.05 else "  ← 평행추세를 기각할 근거 없음"))
es = pd.DataFrame({"period": [c[3:] for c in ev_cols], "b": 100 * m_ev.params[ev_cols].values,
                   "lo": 100 * m_ev.conf_int().loc[ev_cols, 0].values, "hi": 100 * m_ev.conf_int().loc[ev_cols, 1].values})
es = pd.concat([es, pd.DataFrame([{"period": REF, "b": 0, "lo": 0, "hi": 0}])]).sort_values("period")
es["time"] = es.period.map(main.drop_duplicates("period").set_index("period").time)

fig, ax = plt.subplots(figsize=(9, 3.8))
pre = es.period < "202301"
for mask, col in ((pre, MUTED), (~pre, BLUE)):
    e = es[mask]
    ax.errorbar(e.time, e.b, yerr=[e.b - e.lo, e.hi - e.b], fmt="o", color=col, ms=4, capsize=2, lw=1)
ax.axhline(0, color=INK2, lw=0.8)
ax.axvline(2022.95, color=AXIS, lw=1, ls="--")
ax.text(0.01, 0.04, f"처치 전 합동검정 p = {pre_tests['2013~2022']:.3f} (2013~22), {pre_tests['2019~2022']:.3f} (2019~22)",
        transform=ax.transAxes, fontsize=8, color=INK2)
ax.set(title="이벤트 스터디: 노출도 1SD당 ln(청년 ÷ 30~59세) 차이 (기준 2022 하반기, 95% 신뢰구간)", ylabel="%")
fig.tight_layout()
fig.savefig(config.FIGURES / "fig17_이벤트스터디.png")
plt.show()

**4단계 해석**

- 그림(fig16)에서 2022 하반기 이후 상·하 집단 모두 청년이 약 17% 줄었다. 그런데 30~59세는 상 집단에서 +7%, 하 집단에서 −12%로 반대로 움직였다. 그래서 **청년 ÷ 30~59세** 비율로 보면 상 집단이 더 줄어든다.
- 회귀 DiD: 노출도 1SD당 −6.3%로 **유의하다**(p=0.025). 상 vs 하 비교는 −10.9%로 **경계** 수준(p=0.069), 청년 취업자 수(`ly`)만 보면 −3.5%로 **유의하지 않다**(p=0.24).
- **그러나 평행추세가 깨졌다(R1).** 처치 전 계수가 모두 0이라는 가설이 2019~2022(p=0.010)와 2013~2022(p<0.001) 모두에서 **기각된다**. 이벤트 스터디(fig17)에서 처치 전 계수가 이미 위에서 아래로 기울어 있다. 그래서 위 −6.3%를 "AI 이후 생긴 차이"로 읽을 수 없다.

## 5. 다른 설명 찾기 — "AI 말고 이걸 설명할 게 뭐가 있나?"

4단계에서 차이가 보여도, 아래 설명이 남아 있으면 AI 때문이라고 말할 수 없다. 하나씩 검정한다.

| 다른 설명 | 확인 방법 | AI 효과라면 |
|---|---|---|
| 원래 있던 추세 | 직업별 직선 추세(2013~)를 넣고 다시 | 추세를 빼도 남아야 함 |
| 직업 전체의 수요 감소(업종 경기) | 같은 직업의 30~59세에 같은 DiD (R2) | 30~59세는 덜 줄어야 함 |
| 사무직 쏠림 | 같은 대분류 안에서만 비교 | 대분류 안에서도 남아야 함 |
| 한 직업이 결과를 좌우 | 직업을 하나씩 빼고 다시 (R3) | 부호가 바뀌지 않아야 함 |
| 코로나 | 2020~2021 제외 | 남아야 함 |
| 측정(점수 버전, 작은 직업) | 2025 점수, 오차 큰 직업 제외 (R3) | 남아야 함 |
| 같은 크기의 변화가 AI 없이도 있었나 | 가짜 시점 2019 상 (R4) | 가짜 시점에서는 0이어야 함 |
| 청년 인구 구성 변화 | 15~19 / 20~24 / 25~29세 전국 추이 | 특정 연령만의 현상이 아니어야 함 |

In [ ]:
d13 = main.assign(x23=main.z23 * main.post)
report("① 직업별 추세 포함 (2013~)", fit(d13, ["x23"], trend=True), "x23", "5", "원래 추세를 뺀 효과")
report("② 30~59세 ln(취업자) (R2)", fit(d19.assign(), ["x23"], y="la"), "x23", "5", "유의한 음수면 직업 전체 충격")
report("③ 같은 대분류 안에서만 비교", fit(d19, ["x23"], fe="major"), "x23", "5")
report("④ 코로나(2020~21) 제외", fit(d19[~d19.covid], ["x23"]), "x23", "5")
report("⑤ 2025 점수 (R3)", fit(d19, ["x25"]), "x25", "5")
report("⑥ 오차 큰 작은 직업(72·73·82) 제외", fit(d19[~d19.code.isin(["72", "73", "82"])], ["x23"]), "x23", "5")
plc = main[(main.year <= 2022)].assign(xp=lambda d: d.z23 * (d.year >= 2019))
report("⑦ 위약: 가짜 시점 2019 상, 2013~2022 자료 (R4)", fit(plc, ["xp"]), "xp", "5", "유의하면 AI 없이도 같은 변화")

loo = pd.Series({c: 100 * fit(d19[d19.code != c], ["x23"]).params["x23"] for c in occ.code})
print(f"\n⑧ 직업 하나씩 빼기 (R3): 효과 범위 {loo.min():+.2f}% ~ {loo.max():+.2f}%, 부호 바뀜 {int((np.sign(loo) != np.sign(m_main.params['x23'])).sum())}개")
print("   효과를 가장 크게 바꾸는 직업:", ", ".join(f"{occ.set_index('code').name[c]}({v:+.2f})" for c, v in (loo - 100 * m_main.params['x23']).abs().nlargest(3).items()))
results.append({"단계": "5", "분석": "⑧ 직업 하나씩 빼기 (R3)", "효과(%)": loo.median(), "하한": loo.min(), "상한": loo.max(),
                "판정": "부호 유지" if (np.sign(loo) == np.sign(m_main.params["x23"])).all() else "부호 바뀜", "비고": "하한·상한 = 50번 추정의 최소·최대"})

### 5-2. 청년 안의 연령 구성 — 전국 월별

직업별 표에는 15~29세 하나뿐이라 전국 표로 본다. 15~24세 − 15~19세 = 20~24세, 15~29세 − 15~24세 = 25~29세.
통계 검정: 각 연령의 고용률 전년 동월 대비 변화(2023.01 이후 월별)의 평균이 0인지 t검정. 월별 값은 서로 겹쳐서(12개월 차이) 독립이 아니므로 p값은 참고용이다.

In [ ]:
age = {"15~19세": ("10 ",), "20~24세": ("70 ", "10 "), "25~29세": ("75 ", "70 ")}
rows = []
for k, a in age.items():
    pop = series(a[0], "T10") - (series(a[1], "T10") if len(a) > 1 else 0)
    emp = series(a[0], "T30") - (series(a[1], "T30") if len(a) > 1 else 0)
    er = 100 * emp / pop
    y0, y1 = er["2022"].mean(), er.iloc[-12:].mean()
    yoy = er.diff(12)["2023-01":]
    tt_ = stats.ttest_1samp(yoy, 0)
    rows.append({"연령": k, "2022 인구(천)": pop["2022"].mean(), "최근12개월 인구(천)": pop.iloc[-12:].mean(),
                 "2022 고용률": y0, "최근12개월 고용률": y1, "변화(%p)": y1 - y0, "전년동월 변화 평균": yoy.mean(), "p(참고)": tt_.pvalue, "판정": sig(tt_.pvalue)})
print(pd.DataFrame(rows).round(2).to_string(index=False))

### 5-3. 이 자료로 검정할 수 없는 설명

아래는 노출도가 높은 직업(사무·IT·영업)에 같은 시기에 겹쳤을 수 있지만, 공개 직업별 자료에 변수가 없어 검정하지 못했다. 결론에 한계로 적는다.

- 2022~2023년 금리 인상과 그에 따른 IT·스타트업 채용 축소
- 수출 경기와 업종별 업황(제조 사무직 수요)
- 기업의 경력직 선호 같은 채용 관행 변화
- 대학 진학·군 복무 등 청년의 노동시장 진입 시점 변화

MDIS 원자료에는 산업 코드가 있어, 04 노트북에서 산업 구성을 통제한 결과를 따로 봤다.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
fr = pd.DataFrame([r for r in results if r["단계"] in ("4", "5") and pd.notna(r.get("하한"))]).iloc[::-1].reset_index(drop=True)
for i, r in fr.iterrows():
    col = ORANGE if r.p < 0.05 else (MUTED if pd.notna(r.p) else AXIS)
    ax.plot([r.하한, r.상한], [i, i], color=col, lw=1.6)
    ax.plot(r["효과(%)"], i, "o", color=col, ms=5)
ax.axvline(0, color=INK2, lw=0.8)
ax.set_yticks(range(len(fr)), fr.분석, fontsize=7.5)
ax.set(title="4·5단계 효과(%)와 95% 신뢰구간 — 주황은 p < 0.05", xlabel="%")
ax.text(1.0, -0.12, "⑧은 신뢰구간이 아니라 50번 추정의 최소~최대", transform=ax.transAxes, ha="right", va="top", fontsize=7, color=MUTED)
fig.tight_layout()
fig.savefig(config.FIGURES / "fig18_다른설명_검정.png")
plt.show()

**5단계 해석**

| 다른 설명 | 결과 | 유의성 | 뜻 |
|---|---|---|---|
| 원래 있던 추세 | 추세를 빼면 +1.0% | 유의하지 않음 (p=0.50) | **4단계의 −6.3%는 원래 추세로 설명된다** |
| 가짜 시점 2019 상 (R4) | −9.5% | **매우 유의** (p=0.004) | AI가 없던 2019~2022에도 같은 방향, 더 큰 차이가 있었다 |
| 같은 대분류 안에서만 | −2.6% | 유의하지 않음 (p=0.63) | 차이의 대부분이 사무직 대 다른 대분류의 차이에서 나온다 |
| 30~59세 (R2) | +2.9% | 경계 (p=0.051) | 고노출 직업의 중장년은 줄지 않았다. 직업 전체의 수요 감소는 아니다 |
| 코로나 제외, 2025 점수, 작은 직업 제외 | −5.9 ~ −6.4% | 유의 (p=0.02~0.04) | 측정 방식에는 흔들리지 않는다 |
| 직업 하나씩 빼기 | −8.0 ~ −4.1% | 부호 유지 | 특정 직업 하나의 결과가 아니다 |
| 연령 구성 (전국) | 20~24세 고용률 −4.6%p, 25~29세 −0.2%p | 20~24세만 **매우 유의** | 청년 고용률 하락은 20~24세에 몰려 있다. 같은 기간 20~24세 인구도 49만 명 줄었다 |

정리하면, 고노출 직업에서 청년이 중장년보다 더 줄어든 것은 **통계적으로 유의한 사실**이다.
하지만 같은 차이가 AI 이전에도 같은 속도로 진행되고 있었고(위약 검정 유의, 추세 제거 후 유의하지 않음), 그래서 **이 자료로는 AI 때문이라고 말할 수 없다**.

## 6. 결정 기록

위 검정을 판정 규칙(0단계 R1~R4)에 대입한다. 아래 표는 코드가 자동으로 만든다.

In [ ]:
res = pd.DataFrame(results)
res.to_csv(config.FIGURES.parent / "did_eda_results.csv", index=False, encoding="utf-8-sig")
print(res.round(3).to_string(index=False))

b_main, p_main = m_main.params["x23"], m_main.pvalues["x23"]
get = lambda key: next(r for r in results if r["분석"].startswith(key))
rules = {
    "R1 평행추세 (2019~22)": "깨짐 → 직업별 추세 포함 결과를 주로" if pre_tests["2019~2022"] < 0.05 else "기각 안 됨 → 단순 DiD 사용 가능",
    "R1 평행추세 (2013~22)": "깨짐" if pre_tests["2013~2022"] < 0.05 else "기각 안 됨",
    "R2 청년만의 변화": "아님(30~59세도 유의하게 감소)" if get("② ")["p"] < 0.05 and get("② ")["효과(%)"] < 0 else "30~59세는 유의한 감소 없음",
    "R3 견고성": "통과" if p_main < 0.05 and np.sign(get("⑤ ")["효과(%)"]) == np.sign(b_main) and (np.sign(loo) == np.sign(b_main)).all() else "통과 못 함",
    "R4 위약 시점": "유의함 → AI 없이도 같은 변화" if get("⑦ ")["p"] < 0.05 else "유의하지 않음",
}
for k, v in rules.items():
    print(f"{k}: {v}")

### 결정

| 항목 | 결정 | 근거 |
|---|---|---|
| 결과변수 | 주: `lr_y` = ln(청년 ÷ 30~59세), 함께: `ly` | 2-2 품질 통과, 3-2 로그 변환 근거. 직업 전체 충격을 거르려면 비율이 필요 |
| 처치 | ILO 2023 점수, 연속값(1SD). 2025 점수는 검증용 | 3-1 순위 상관 0.97 |
| 표본 | 50개 직업, 2022년 청년 취업자 가중, 직업 단위 군집 | 2-3 작은 직업을 빼도 결과 같음(5단계 ⑥) |
| 기간 | 처치 전 2019~2022, 처치 후 2023~2026. 추세 추정에는 2013년부터 | 2-2 자료 품질은 2013년부터 문제없음 |
| **모형** | **단순 DiD를 인과 결과로 쓰지 않는다.** 직업별 추세를 넣은 DiD를 주 결과로 쓰고, 단순 DiD는 "관찰된 차이"로만 보고한다 | R1: 평행추세가 두 기간 모두 기각됨 |
| 다음 분석 | ① 추세 포함 이벤트 스터디로 2023년 이후 추세에서 벗어났는지 확인 ② 같은 대분류 안 비교를 기본 견고성 검정으로 추가 | 5단계 ①·③ |

### 발표에서 쓸 수 있는 문장

- 쓸 수 있음: "AI 노출이 큰 직업에서 청년 취업자가 중장년보다 더 줄었다(노출도 1SD당 약 −6%, 통계적으로 유의)."
- 쓸 수 있음: "그러나 이 차이는 생성형 AI 이전(2013~2022)부터 같은 방향으로 진행되고 있었다. 기존 추세를 빼면 차이가 사라진다."
- 쓸 수 없음: "AI가 고노출 직업의 청년 일자리를 6% 줄였다."

효과가 "없다"는 뜻은 아니다. 추세를 뺀 결과의 95% 신뢰구간은 −1.8% ~ +3.8%라서, 그보다 큰 AI 효과는 이 자료와 맞지 않고 그보다 작은 효과는 배제할 수 없다.

## 7. 후속 분석: 최근에 꺾였나

5단계에서 AI 이후 3년 반을 **평균 내면** 원래 추세선 위에 있었다. 하지만 평균은 "처음엔 추세대로였다가 최근에만 더 떨어진" 경우를 가릴 수 있다.
그래서 두 가지를 본다.

1. **반기별 벗어남**: AI 이전 추세를 연장한 선(예측선)보다 반기마다 얼마나 아래에 있나
2. **기울기 변화**: AI 이후 떨어지는 속도가 그 전보다 빨라졌나 (계수 하나라서 검정력이 더 좋다)

방법: 노출도 1SD당 차이에 직선 추세(`z23 × 시간`)를 넣는다. 처치 후 반기마다 따로 더미를 두면, 추세 기울기는 처치 전 자료로만 정해지고
더미 계수가 곧 "예측선에서 벗어난 정도"가 된다.

**예측선을 어느 기간으로 그리느냐가 결과를 좌우할 수 있다.** 2013~2022 전체로 그리면 기울기가 가파르고, 2019~2022로만 그리면 완만하다.
둘 다 보고, 결과가 같은 방향인지 확인한다.

In [ ]:
post_k = [k for k in PERIODS if k >= "202301"]
main["tz"] = main.z23 * main.t                    # 노출도 × 시간(년): 원래 추세
main["tz_post"] = main.z23 * main.t.clip(lower=0)  # AI 이후에만 늘어나는 시간: 기울기 변화
for k in post_k:
    main[f"dv_{k}"] = main.z23 * (main.period == k)
dv_cols = [f"dv_{k}" for k in post_k]
WIN = {"2013~2022 추세": main, "2019~2022 추세": main[main.year >= 2019]}

dev, slope_rows = [], []
for wname, d in WIN.items():
    m = fit(d, ["tz"] + dv_cols)
    ci = m.conf_int()
    for c in dv_cols:
        dev.append({"추세 기간": wname, "period": c[3:], "벗어남(%)": 100 * m.params[c], "하한": 100 * ci.loc[c, 0], "상한": 100 * ci.loc[c, 1], "p": m.pvalues[c]})
    p_all = float(m.f_test(", ".join(f"{c} = 0" for c in dv_cols)).pvalue)
    last2 = m.t_test(f"{dv_cols[-2]} + {dv_cols[-1]} = 0")
    b2, p2 = float(np.squeeze(last2.effect)) / 2, float(last2.pvalue)
    print(f"[{wname}] 처치 전 기울기 {100 * m.params['tz']:+.2f}%/년 (1SD당)")
    avg = m.t_test(" + ".join(dv_cols) + " = 0")
    b_avg, p_avg = float(np.squeeze(avg.effect)) / len(dv_cols), float(avg.pvalue)
    print(f"   7개 반기 벗어남이 모두 0인가(합동 F검정): p={p_all:.3f} → {sig(p_all)}  ← 방향은 알려주지 않음")
    print(f"   7개 반기 평균 벗어남: {100 * b_avg:+.2f}%, p={p_avg:.3f} → {sig(p_avg)}  ← 음수면 평균적으로 예측선보다 아래")
    print("   반기별로 유의한(p<0.05) 벗어남:", [c[3:] for c in dv_cols if m.pvalues[c] < 0.05] or "없음")
    results.append({"단계": "7", "분석": f"AI 이후 평균 벗어남 ({wname})", "효과(%)": 100 * b_avg, "p": p_avg, "판정": sig(p_avg), "비고": "예측선 대비"})
    print(f"   최근 2개 반기(2025 하·2026 상) 평균 벗어남: {100 * b2:+.2f}%, p={p2:.3f} → {sig(p2)}")
    results.append({"단계": "7", "분석": f"최근 2개 반기 벗어남 ({wname})", "효과(%)": 100 * b2, "p": p2, "판정": sig(p2), "비고": "예측선 대비"})
    results.append({"단계": "7", "분석": f"AI 이후 반기별 벗어남 합동검정 ({wname})", "p": p_all, "판정": sig(p_all)})
dev = pd.DataFrame(dev)
print("\n" + dev.pivot(index="period", columns="추세 기간", values="벗어남(%)").round(2).to_string())

### 7-2. 기울기 변화 — 떨어지는 속도가 빨라졌나

같은 검정을 청년 ÷ 30~59세(`lr_y`), 청년 취업자(`ly`), 30~59세 취업자(`la`)에 각각 한다.
`lr_y` = `ly` − `la`라서, 비율이 꺾였다면 그게 청년이 줄어서인지 중장년이 늘어서인지 나눠 볼 수 있다.
계수는 "AI 이후 기울기 − AI 이전 기울기", 단위는 노출도 1SD당 %/년이다.

In [ ]:
for wname, d in WIN.items():
    for y, lab in (("lr_y", "청년 ÷ 30~59세"), ("ly", "청년 취업자"), ("la", "30~59세 취업자")):
        m = fit(d, ["tz", "tz_post"], y=y)
        r = report(f"기울기 변화 {lab} ({wname})", m, "tz_post", "7", "AI 이후 − 이전, %/년")
        slope_rows.append({"추세 기간": wname, "결과": lab, "이전 기울기": 100 * m.params["tz"], "변화": r["효과(%)"], "p": r["p"]})
print("\n" + pd.DataFrame(slope_rows).round(3).to_string(index=False))

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1.25, 1]})
e = es.copy()
a1.errorbar(e.time, e.b, yerr=[e.b - e.lo, e.hi - e.b], fmt="o", color=INK2, ms=3.5, capsize=2, lw=0.9, label="실제 (반기별 차이)")
for (wname, lo_year), col in zip((("2013~2022 추세", 2013), ("2019~2022 추세", 2019)), (ORANGE, BLUE)):
    pre_pts = e[(e.period < "202301") & (e.time >= lo_year)]
    k_, c_ = np.polyfit(pre_pts.time, pre_pts.b, 1)
    xs = np.array([pre_pts.time.min(), e.time.max()])
    a1.plot(xs, k_ * xs + c_, color=col, lw=1.4, ls="--", label=f"예측선: {wname}")
a1.axvline(2022.95, color=AXIS, lw=1, ls=":")
a1.axhline(0, color=AXIS, lw=0.8)
a1.set(title="노출도 1SD당 ln(청년 ÷ 30~59세) 차이와 AI 이전 추세의 연장선", ylabel="% (기준 2022 하반기)")
a1.legend(frameon=False, fontsize=7.5, loc="upper right")

for (wname, g), col, off in zip(dev.groupby("추세 기간", sort=False), (ORANGE, BLUE), (-0.06, 0.06)):
    tt_ = g.period.map(main.drop_duplicates("period").set_index("period").time) + off
    a2.errorbar(tt_, g["벗어남(%)"], yerr=[g["벗어남(%)"] - g.하한, g.상한 - g["벗어남(%)"]], fmt="o", color=col, ms=4, capsize=2, lw=1, label=wname)
a2.axhline(0, color=INK2, lw=0.8)
a2.set(title="예측선에서 벗어난 정도 (95% 신뢰구간)", ylabel="%", xticks=[2023.3, 2024.3, 2025.3, 2026.3], xticklabels=["2023", "2024", "2025", "2026"])
a2.legend(frameon=False, fontsize=7.5, loc="lower left")
a2.text(0.98, 0.97, "0보다 아래 = 원래 추세보다 더 떨어짐", transform=a2.transAxes, ha="right", va="top", fontsize=7, color=MUTED)
fig.tight_layout()
fig.savefig(config.FIGURES / "fig19_추세대비_벗어남.png")
plt.show()

res = pd.DataFrame(results)
res.to_csv(config.FIGURES.parent / "did_eda_results.csv", index=False, encoding="utf-8-sig")

**7단계 해석** (2026-11-24 실행 기준)

| 질문 | 2013~2022 추세로 예측 | 2019~2022 추세로 예측 |
|---|---|---|
| 반기 중 예측선에서 유의하게 벗어난 때가 있나 | 없음 | 없음 |
| AI 이후 평균 벗어남 | +0.9% (p=0.51, 유의하지 않음) | −1.9% (p=0.42, 유의하지 않음) |
| 최근 2개 반기 평균 벗어남 | +0.3% (p=0.89, 유의하지 않음) | −3.6% (p=0.20, 유의하지 않음) |
| 떨어지는 속도의 변화 (청년 ÷ 30~59세) | +0.2%/년 (p=0.66) | −0.9%/년 (p=0.16) |

- **최근에 꺾였다는 통계적 근거는 없다.** 어느 예측선을 쓰든 평균·최근·기울기 모두 유의하지 않다.
- 다만 방향은 예측선에 따라 다르다. 완만한 2019~2022 추세를 기준으로 하면 2026 상반기가 예측선보다 5% 아래(p=0.10)라서, **"약하게 꺾이는 조짐"으로 볼 여지는 있다.** 한두 반기 자료가 더 쌓여야 판단할 수 있다.
- 합동 F검정이 유의한 것(p=0.002)은 방향이 일정한 하락이 아니라, 2024 하반기에는 위로, 2026 상반기에는 아래로 **오르내린 움직임** 때문이다. 평균 벗어남 검정이 유의하지 않은 것이 그 증거다. 군집이 50개뿐인데 7개를 한꺼번에 검정하면 p값이 실제보다 작게 나오기 쉽다는 점도 감안한다.
- 2019~2022 기준에서 꺾임처럼 보이는 부분(−0.9%/년)을 나눠 보면, 청년 취업자는 오히려 덜 줄었고(+0.3%/년, 유의하지 않음) **같은 직업의 30~59세가 더 빨리 늘었다**(+1.2%/년, p=0.044, 유의). 즉 "청년이 밀려났다"보다는 "고노출 직업에서 경력자가 늘었다"에 가깝다.

**발표에서 쓸 수 있는 문장**: "AI 이후에도 고노출 직업의 청년 비중은 원래 추세대로 줄고 있다. 최근 더 빨리 줄어든다는 통계적 근거는 아직 없다. 다만 가장 최근 반기는 추세보다 낮아서 앞으로 지켜볼 필요가 있다."